# 🎙️ Atomic Step 6: Speaker-Wise Transcription (Local Whisper-LoRA) & Insights Extraction (Gemini API)

This notebook runs Gurmukhi script Punjabi transcription locally using a local fine-tuned `openai/whisper-large-v3-turbo` model wrapped with custom PEFT/LoRA adapter weights (`Garden2006/whisper-large-v3-turbo-gurmukhi-lora`). 

Then, it queries the **Gemini API** (`gemini-3.1-flash-lite` or similar) to transliterate the Punjabi text into Devanagari script (Hindi/Punjabi-in-Devanagari) and extract structured insights, key questions, and knowledge tables.

In [ ]:
# 1. Uninstall incompatible torchao to prevent conflicts
!pip uninstall -y -q torchao

# 2. Install Transformers, PEFT, Accelerate, noisereduce, and Gemini SDK
!pip install -q transformers peft accelerate google-generativeai librosa soundfile --prefer-binary
print("[SUCCESS] Dependencies installed!")

In [ ]:
import torch
from google.colab import userdata

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {device}")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("[SUCCESS] Google Drive mounted.")
except Exception:
    print("[INFO] Already mounted or skipped.")

## ⚙️ Parameters

In [ ]:
# @markdown ### 📁 Audio & Timeline Inputs
cleaned_audio_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Cleaned_Audio/MarauliKhurad1_cleaned.wav" # @param {type:"string"}
refined_timeline_json_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Diarization_Outputs/MarauliKhurad1/MarauliKhurad1_timeline.json" # @param {type:"string"}
transcription_output_folder = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Diarization_Transcripts/" # @param {type:"string"}

# @markdown ### 🤖 Model Configs
whisper_base_model = "openai/whisper-large-v3-turbo" # @param {type:"string"}
whisper_lora_repo = "Garden2006/whisper-large-v3-turbo-gurmukhi-lora" # @param {type:"string"}

# @markdown ### 🔮 Gemini API Integration
run_insights = True # @param {type:"boolean"}
gemini_api_key = "" # @param {type:"string"}
gemini_model_name = "gemini-3.1-flash-lite" # @param {type:"string"}

if not os.path.exists(cleaned_audio_path):
    print(f"[ERROR] Cleaned audio not found at: '{cleaned_audio_path}'")
elif not os.path.exists(refined_timeline_json_path):
    print(f"[ERROR] Refined timeline JSON not found at: '{refined_timeline_json_path}'")
else:
    audio_filename = os.path.basename(cleaned_audio_path)
    audio_name_only = audio_filename.replace("_cleaned.wav", "").replace(".wav", "")
    os.makedirs(transcription_output_folder, exist_ok=True)
    print(f"[SUCCESS] Validated paths. Output transcripts will be saved in: {transcription_output_folder}")

## 🎙️ Step 1: Run Local Whisper-LoRA Transcription

In [ ]:
import soundfile as sf
import librosa
import json
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from peft import PeftModel

if os.path.exists(cleaned_audio_path) and os.path.exists(refined_timeline_json_path):
    # Create subfolder for output transcripts
    specific_transcript_folder = os.path.join(transcription_output_folder, audio_name_only)
    os.makedirs(specific_transcript_folder, exist_ok=True)
    
    # Load timeline JSON
    with open(refined_timeline_json_path, "r", encoding="utf-8") as f:
        speaker_segments = json.load(f)
        
    # Load cleaned audio
    print(f"Loading cleaned audio file for transcription: '{cleaned_audio_path}'")
    y, sr = librosa.load(cleaned_audio_path, sr=16000, mono=True)
    
    print(f"\n--- Transcribing Segments via Local Whisper LoRA Model ---")
    diarized_transcript_entries = []
    speaker_texts = {}
    
    # Time formatting helper
    def format_time(seconds):
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        millis = int((seconds - int(seconds)) * 10)
        if hours > 0:
            return f"{hours:02d}:{minutes:02d}:{secs:02d}.{millis:01d}"
        else:
            return f"{minutes:02d}:{secs:02d}.{millis:01d}"
            
    # Whisper model setup
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    
    print(f"Loading processor and base model: {whisper_base_model}...")
    try:
        processor = AutoProcessor.from_pretrained(whisper_lora_repo, language="punjabi", task="transcribe")
    except Exception:
        processor = AutoProcessor.from_pretrained(whisper_base_model, language="punjabi", task="transcribe")
        
    base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        whisper_base_model,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        device_map="auto" if device == "cuda" else None
    )
    
    print(f"Merging trained LoRA adapters from {whisper_lora_repo}...")
    model = PeftModel.from_pretrained(base_model, whisper_lora_repo)
    model.to(device)
    model.eval()
    print("[SUCCESS] Whisper model with LoRA adapters loaded successfully!")
    
    # Run transcription loop
    for idx, entry in enumerate(speaker_segments):
        start_sec = entry['start']
        end_sec = entry['end']
        speaker = entry['speaker']
        
        duration = end_sec - start_sec
        if duration < 0.3:
            continue
            
        # Slice chunk in-memory
        start_sample = int(start_sec * sr)
        end_sample = int(end_sec * sr)
        chunk = y[start_sample:end_sample]
        
        # Extract features
        input_features = processor(
            chunk,
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features.to(device).to(torch_dtype)
        
        # Generate tokens
        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                language="punjabi",
                task="transcribe"
            )
            
        # Decode tokens
        text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
        
        if text:
            time_str = f"[{format_time(start_sec)} - {format_time(end_sec)}]"
            print(f"{time_str} {speaker}: {text}")
            
            diarized_transcript_entries.append({
                "time": time_str,
                "speaker": speaker,
                "text": text
            })
            
            if speaker not in speaker_texts:
                speaker_texts[speaker] = []
            speaker_texts[speaker].append(text)
            
    # Export outputs
    if diarized_transcript_entries:
        txt_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_diarized_transcript.txt")
        md_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_diarized_transcript.md")
        json_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_diarized_transcript.json")
        
        # Write Text File
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(f"Diarized Transcript for: {audio_filename}\n")
            f.write("=" * 60 + "\n\n")
            for entry in diarized_transcript_entries:
                f.write(f"{entry['time']} {entry['speaker']}: {entry['text']}\n")
                
        # Write Markdown File
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(f"# 🎙️ Diarized Transcript: {audio_filename}\n\n")
            f.write(f"Generated using local Whisper model ({whisper_base_model}) with LoRA adapters ({whisper_lora_repo}).\n\n")
            for entry in diarized_transcript_entries:
                f.write(f"> **{entry['speaker']}** `{entry['time']}`  \n")
                f.write(f"> {entry['text']}\n\n")
                
        # Write JSON File
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(diarized_transcript_entries, f, indent=4, ensure_ascii=False)
            
        # Write Speaker Segregated Files
        for speaker, texts in speaker_texts.items():
            spk_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_{speaker}_transcript.txt")
            with open(spk_path, "w", encoding="utf-8") as f:
                f.write("\n".join(texts))
                
        print(f"\n[SUCCESS] Transcription files exported successfully to: '{specific_transcript_folder}'")
    else:
        print("[WARNING] No speech content could be transcribed.")

## 🔮 Step 2: Run Devanagari Conversion & Insights Extraction (Gemini API)

In [ ]:
import google.generativeai as genai
from google.colab import userdata
from IPython.display import Markdown, display

if run_insights and 'diarized_transcript_entries' in locals() and diarized_transcript_entries:
    # Retrieve API key
    api_key = gemini_api_key.strip()
    if not api_key:
        try:
            api_key = userdata.get('GEMINI_API_KEY')
            print("[INFO] Retrieved GEMINI_API_KEY from Colab Secrets.")
        except Exception:
            pass
            
    if not api_key:
        print("[ERROR] Gemini API Key not found. Please provide it in the form parameter or configure 'GEMINI_API_KEY' in your Colab Secrets.")
    else:
        # Configure Gemini
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel(gemini_model_name)
        
        # Format transcript text
        transcript_text = ""
        for entry in diarized_transcript_entries:
            transcript_text += f"{entry['time']} {entry['speaker']}: {entry['text']}\n"
            
        prompt = f"""
You are an expert AI assistant specializing in natural language processing, translation, and analysis.

You are provided with a sequence-based speaker-diarized Punjabi transcript (written in Gurmukhi script). Please perform the following two tasks:

1. **Devanagari Conversion**: Translate/transliterate the Gurmukhi script Punjabi transcript into Devanagari script (Hindi/Punjabi-in-Devanagari, ensuring it is natural and easy to read for Hindi speakers). Keep the speaker labels and timestamps intact.
2. **Insights, Questions, and Knowledge Extraction**: 
   - **Key Insights**: Extract the main takeaways and core ideas discussed by the speakers.
   - **Questions & Inquiries**: Note any questions raised or problems mentioned by the speakers that require action or follow-up.
   - **Knowledge Dictionary/Grid**: Draw up a structured markdown table with key technical words, local references, village terms, or agricultural terms used, detailing their meanings.

Format your entire response as a single, premium Markdown document with clear headers and layout.

--- Transcript ---
{transcript_text}
"""
        
        print("Sending request to Gemini API...")
        try:
            response = model.generate_content(prompt)
            insights_md = response.text
            
            # Save file
            insights_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_devanagari_insights.md")
            with open(insights_path, "w", encoding="utf-8") as f:
                f.write(insights_md)
                
            print(f"\n[SUCCESS] Devanagari translation & insights successfully stored at: '{insights_path}'")
            print("\n--- Insights Preview ---")
            display(Markdown(insights_md))
        except Exception as e:
            print(f"[ERROR] Gemini API request failed: {e}")